In [10]:
import polars as pl
from pathlib import Path
import jellyfish
from nltk.corpus import names
import nltk
import re
import altair as alt

In [11]:
# root = Path("../..")
# parquet = root / 'data/jails-data/SERVER/run 0728/output/jails_pdfs_cleanned.parquet'
# person_parquet = root / 'data/jails-data/SERVER/run 0728/output/jails_person_records.parquet'
# df_cleaned = pl.read_parquet(parquet)
# people_parquet = root / 'data/jails-data/SERVER/run 0728/output/jails_person_records.parquet'
# people = pl.read_parquet(people_parquet)
df_cleaned = pl.read_parquet("jails_pdfs_cleanned.parquet")
people = pl.read_parquet("jails_person_records.parquet")

In [12]:
merge = people.join(df_cleaned, on="Report ID")

merge.write_excel("tuesday comparison")

In [13]:
df_cleaned= df_cleaned.with_columns(
    pl.when(pl.col("Cleaned Facility Name").str.contains("Peoria"))
    .then(pl.lit("Peoria County Jail"))
    .when(pl.col("Cleaned Facility Name").str.contains("Adams"))
    .then(pl.lit("Adams County Jail"))
    .when(pl.col("Cleaned Facility Name").str.contains("Dupage"))
    .then(pl.lit("Dupage County Jail"))
    .when(pl.col("Cleaned Facility Name").str.contains("Kane"))
    .then(pl.lit("Kane County Jail"))
    .when(pl.col("Cleaned Facility Name").str.contains("Lake County"))
    .then(pl.lit("Lake County Jail"))
    .when(pl.col("Cleaned Facility Name").str.contains("Sangamon"))
    .then(pl.lit("Sangamon County Jail"))
    .when(pl.col("Cleaned Facility Name").str.contains("Vill County"))
    .then(pl.lit("Will County Jail"))
    .when(pl.col("Cleaned Facility Name").str.contains("Williamson"))
    .then(pl.lit("Williamson County Jail"))
    .when(pl.col("Cleaned Facility Name").str.contains("Winnebago"))
    .then(pl.lit("Williamson County Jail"))
    .otherwise(pl.col("Cleaned Facility Name"))
    .alias("Cleaned Facility Name")
)

In [14]:
total_occurrence_result = (
    df_cleaned
    .with_columns(
        pl.col("Cleaned Occurrences").str.split("; ").alias("Cleaned Occurrences")
    )
    .explode("Cleaned Occurrences")
    .filter(pl.col("Cleaned Occurrences").is_not_null())
    .select(
        pl.col("Cleaned Occurrences").value_counts(sort=True)
    )
    .unnest("Cleaned Occurrences")
    .head(12)
)

total_occurrence_result = total_occurrence_result.filter(~pl.col("Cleaned Occurrences").is_in(["Error, no occurrence found"]))

total_occurrence_chart = alt.Chart(total_occurrence_result).mark_bar().encode(
    x=alt.X("count:Q", title="Count"),
    y=alt.Y("Cleaned Occurrences:N", title="Occurrence").sort('-x')
).properties(
    title="Most Common Occurrences Across All Jails"
)

total_occurrence_chart = alt.Chart(total_occurrence_result).mark_bar(
    color="#54728d"
).encode(
    x=alt.X('count:Q', title='Number of Cases'),
    y=alt.Y('Cleaned Occurrences:N', sort='-x', title='Occurrence'),
    tooltip=[
        alt.Tooltip('Cleaned Occurrences:N', title='Occurrence'),
        alt.Tooltip('count:Q', format=',d', title='Case Count')
    ]
).properties(
    width=600,
    height=400,
    title='Most Common Occurrences Across All Jails'
)

# Footnote as a separate text chart (keep your note)
footnote = alt.Chart(pl.DataFrame({
    "text": ["Note: Occurrences may be concurrent. Missing 2,027 Occurrences (7.3%)"],
    "x": [0],
    "y": [0]
})).mark_text(
    align="left",
    baseline="bottom",
    fontSize=10,
    dx=0,
    dy=20,
    color="gray"
).encode(
    x=alt.value(-50),
    y=alt.value(440),
    text="text:N"
)

# Layer chart and footnote, remove box
total_occurrence_chart = alt.layer(
    total_occurrence_chart,
    footnote
).configure_view(
    stroke=None
)

In [15]:
total_occurrence_chart

alt.LayerChart(...)

In [34]:
top_counties = (
    df_cleaned
    .select(["Cleaned Facility Name", "Cleaned Occurrences"])
    .with_columns(
        pl.col("Cleaned Occurrences").str.split("; ").alias("Cleaned Occurrences")
    )
    .explode("Cleaned Occurrences")
    .with_columns(
        pl.col("Cleaned Occurrences").str.strip_chars().alias("Cleaned Occurrences")
    )
    .filter(pl.col("Cleaned Occurrences").is_not_null())
)

total_reports = (
    top_counties
    .group_by("Cleaned Facility Name")
    .agg(pl.len().alias("total_reports"))
)

def get_county_occurrences(current_jail):
    filtered_df = top_counties.filter(pl.col("Cleaned Facility Name") == current_jail)
    current_jail_result = (
    filtered_df.select(
        pl.col("Cleaned Occurrences").value_counts(sort=True)
    )
    .unnest("Cleaned Occurrences")
    .head(12)
    )
    current_jail_result = current_jail_result.filter(~pl.col("Cleaned Occurrences").is_in(["Error", "no occurrence found"]))
    current_jail_chart = alt.Chart(current_jail_result).mark_bar(
    color="#54728d").encode(
    x=alt.X('count:Q', title='Number of Cases'),
    y=alt.Y('Cleaned Occurrences:N', sort='-x', title='Occurrence'),
    tooltip=[
        alt.Tooltip('Cleaned Occurrences:N', title='Occurrence'),
        alt.Tooltip('count:Q', format=',d', title='Case Count')
    ]
    ).properties(
    width=600,
    height=400,
    title=current_jail
    )
    return current_jail_chart

def get_county_occurrences_percent(current_jail):
    filtered_df = top_counties.filter(pl.col("Cleaned Facility Name") == current_jail)
    current_jail_result = (
        filtered_df.select(
            pl.col("Cleaned Occurrences").value_counts(sort=True)
        )
        .unnest("Cleaned Occurrences")
        .filter(~pl.col("Cleaned Occurrences").is_in(["Error", "no occurrence found"]))
    )
    total_count = current_jail_result["count"].sum()
    current_jail_result = current_jail_result.with_columns(
        (pl.col("count") / total_count * 100).alias("percentage")
    )
    current_jail_result = current_jail_result.sort("count", descending=True).head(10)
    current_jail_chart = alt.Chart(current_jail_result).mark_bar(
    color="#54728d").encode(
    x=alt.X('count:Q', title='Number of Cases'),
    y=alt.Y('Cleaned Occurrences:N', sort='-x', title='Occurrence'),
    tooltip=[
        alt.Tooltip('Cleaned Occurrences:N', title='Occurrence'),
        alt.Tooltip('count:Q', format=',d', title='Case Count')
    ]
    ).properties(
    width=600,
    height=400,
    title=current_jail
    )

    return current_jail_chart

cook_county = get_county_occurrences("Cook County Department Of Corrections")

kane_county = get_county_occurrences("Kane County Jail")

madison_county = get_county_occurrences("Madison County Sheriffs Office")

lake_county = get_county_occurrences("Lake County Jail")

peoria = get_county_occurrences("Peoria County Jail")

In [35]:
cook_county

alt.Chart(...)

In [18]:
pattern = r"\bRestraints\b"

restraints_table = df_cleaned.filter(
    pl.col("Cleaned Occurrences").str.contains(pattern)
)

restraints_table = (
    restraints_table.select(
        pl.col("Cleaned Facility Name").value_counts(sort=True)
    )
    .unnest("Cleaned Facility Name")
    .head(21)
)

restraints_table = restraints_table.filter(pl.col("Cleaned Facility Name") != "")

restraints_chart = alt.Chart(restraints_table).mark_bar(
    color="#5a8664").encode(
    x=alt.X("count:Q", title="Count"),
    y=alt.Y("Cleaned Facility Name:N", title="Jail").sort('-x'),
    tooltip=[
        alt.Tooltip("Cleaned Facility Name:N", title="Jail"),
        alt.Tooltip("count:Q", format=",d", title="Case Count")
    ]
).properties(
    width=600,
    height=400,
    title="Top Jails for Restraint Usage"
)

# Footnote as a separate text chart
footnote = alt.Chart(pl.DataFrame({
    "text": ["Note: Missing 1,119 County Names (4.33%)"],
    "x": [0],
    "y": [0]
})).mark_text(
    align="left",
    baseline="bottom",
    fontSize=10,
    dx=0,
    dy=20,  # distance below the main chart area
    color="gray"
).encode(
    x=alt.value(0),
    y=alt.value(440),  # just below the chart (chart height + some offset)
    text="text:N"
)

# Use layering to draw the footnote over/under the chart
restraints_chart = alt.layer(
    restraints_chart,
    footnote
).configure_view(
    stroke=None  # removes box around the chart
)

In [19]:
restraints_table = restraints_table.filter(
    pl.col("Cleaned Facility Name") != "null"
).join(total_reports, on="Cleaned Facility Name")

restraints_per_jail = (
    top_counties
    .filter(pl.col("Cleaned Occurrences") == "Restraints Used")
    .group_by("Cleaned Facility Name")
    .agg(pl.len().alias("restraint_count"))
)

merged = (
    restraints_per_jail
    .join(total_reports, on="Cleaned Facility Name")
    .with_columns(
        (pl.col("restraint_count") / pl.col("total_reports") * 100).alias("perc")
    )
    .filter(pl.col("total_reports") >= 100)
)

In [20]:
restraints_perc_chart = alt.Chart(merged.to_pandas()).mark_bar().encode(
    x=alt.X("perc:Q", title="Percentage"),
    y=alt.Y("Cleaned Facility Name:N", title="Jail").sort('-x')
).properties(
    width=600,
    height=400,
    title="Top Jails for Restraint Usage as a Percent of Reports"
)

# Footnote as a separate text chart
footnote = alt.Chart(pl.DataFrame({
    "text": ["Note: Only Jails with over 100 Reports Total"],
    "x": [0],
    "y": [0]
})).mark_text(
    align="left",
    baseline="bottom",
    fontSize=10,
    dx=0,
    dy=20,  # distance below the main chart area
    color="gray"
).encode(
    x=alt.value(0),
    y=alt.value(440),  # just below the chart (chart height + some offset)
    text="text:N"
)

# Use layering to draw the footnote over/under the chart
restraints_perc_chart = alt.layer(
    restraints_perc_chart,
    footnote
).configure_view(
    stroke=None  # removes box around the chart
)

In [21]:
pattern = r"\bSuicide\b"

suicide_table = top_counties.filter(
    pl.col("Cleaned Occurrences").str.contains(pattern)
)

suicide_table = (
    suicide_table.select(
        pl.col("Cleaned Facility Name").value_counts(sort=True)
    )
    .unnest("Cleaned Facility Name")
    .head(11)
)

suicide_table_table = suicide_table.filter(pl.col("Cleaned Facility Name") != "")

suicide_chart = alt.Chart(suicide_table).mark_bar().encode(
    x=alt.X("count:Q", title="Count"),
    y=alt.Y("Cleaned Facility Name:N", title="Jail").sort('-x')
).properties(
    width=600,
    height=400,
    title="Top Jails for Self Harm among Detainees"
)

In [22]:
merge = people.join(df_cleaned, on="Report ID")

top_people = (
    people.select(
        pl.col("Name").value_counts(sort=True)
    )
    .unnest("Name")
    .head(10)
    )